# Exploratory Data Analysis — Default of Credit Card Clients

This notebook is for **exploration and visualisation only**. All production
logic (ingestion, validation, cleaning, feature engineering, preprocessing)
lives in the `credit_risk` package and is imported here, so the notebook can
never drift from what the model actually does.

Run the pip-install step first: `pip install -e .`

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from credit_risk.config import get_config
from credit_risk.data.ingestion import dataset_fingerprint, load_raw_data
from credit_risk.data.preprocessing import clean_data
from credit_risk.data.validation import validate_data

pd.set_option("display.width", 140)
cfg = get_config()

## 1. Load and validate

`validate_data` runs the same structural checks used in the training pipeline.

In [ ]:
raw = load_raw_data()
report = validate_data(raw)
print(raw.shape)
print("fingerprint:", dataset_fingerprint(raw))
print("default rate: %.3f" % report.default_rate)
print("out-of-codebook:", report.out_of_codebook)
raw.dtypes

## 2. Target distribution

The classes are mildly imbalanced (~22% default), which is why every model is
evaluated with PR-AUC, precision, and recall rather than accuracy.

In [ ]:
target = cfg.columns.target
ax = raw[target].value_counts().sort_index().plot.bar(
    color=["#4C72B0", "#C44E52"], title="Target distribution"
)
ax.set_xticklabels(["No default (0)", "Default (1)"], rotation=0)
plt.show()

## 3. Default rate by categorical attribute

`sex`, `education`, and `marriage` are **not** used as model features; `sex` and
`marriage` are retained only for the fairness audit. They are shown here purely
to characterise the data.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, column in zip(axes, ["sex", "education", "marriage"]):
    rates = raw.groupby(column)[target].mean()
    ax.bar(rates.index.astype(str), rates.values, color="#C44E52")
    ax.set_title(f"Default rate by {column}")
    ax.set_ylabel("default rate")
plt.tight_layout()
plt.show()

## 4. Numeric distributions and outliers

Bill and payment amounts are heavily right-skewed with multi-million NT$
outliers; the pipeline winsorises them at training-set percentiles.

In [ ]:
raw[["limit_bal", "age", "bill_amt1", "pay_amt1"]].describe()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, column in zip(axes, ["limit_bal", "age", "bill_amt1", "pay_amt1"]):
    raw[column].clip(raw[column].quantile(0.01), raw[column].quantile(0.99)).hist(ax=ax, bins=40)
    ax.set_title(column)
plt.tight_layout()
plt.show()

## 5. Engineered features vs. target

The engineered delinquency features separate the classes cleanly — an expected
signal in credit default. This is association, not causation.

In [ ]:
from credit_risk.features.engineering import engineer_features

engineered = engineer_features(clean_data(raw))
engineered.groupby(target)[[
    "payment_delay_rate", "max_delay_months", "credit_utilization", "payment_to_bill_ratio"
]].mean()

## 6. Leakage notes

* The dataset is a cross-sectional snapshot with **no temporal ordering**, so a
  stratified random split is correct and duplicate-client leakage cannot occur
  (every `id` is unique).
* All fitted statistics (medians, winsorisation bounds, scaler, encoder) are
  fit on the training split only — see `credit_risk.data.preprocessing`.
* `sex` and `marriage` are excluded from the model and used only for auditing.